In [1]:
!pip install yt-dlp moviepy azure-cognitiveservices-speech azure-ai-textanalytics requests

## IMPORTS

In [2]:
import yt_dlp
from moviepy.editor import VideoFileClip
import azure.cognitiveservices.speech as speechsdk
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
import requests
import time
import os

## AZURE CONFIG

In [ ]:
AZURE_SPEECH_KEY = "YOUR_AZURE_SPEECH_KEY"
AZURE_SPEECH_REGION = "francecentral"

AZURE_LANGUAGE_KEY = "YOUR_AZURE_LANGUAGE_KEY"
AZURE_LANGUAGE_ENDPOINT = "https://text-summarization-project.cognitiveservices.azure.com/"
TRANSLATOR_KEY = "YOUR_TRANSLATOR_KEY"
TRANSLATOR_ENDPOINT = "https://api.cognitive.microsofttranslator.com"
TRANSLATOR_REGION = "francecentral"

## DOWNLOAD VIDEO

In [4]:
def download_video(url):
    ydl_opts = {
        'outtmpl': 'video.mp4',
        'format': 'best',
        'http_headers': {
            'User-Agent': 'Mozilla/5.0'
        }
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    print("✅ Video downloaded!")



## EXTRACT AUDIO

In [5]:
def extract_audio(video_path, audio_path):
    video = VideoFileClip(video_path)
    video.audio.write_audiofile(
        audio_path,
        fps=16000,
        codec='pcm_s16le'
    )
    print("✅ Audio extracted!")


## SPEECH TO TEXT (LONG AUDIO)

In [6]:
def speech_to_text_long(audio_file):
    speech_config = speechsdk.SpeechConfig(
        subscription=AZURE_SPEECH_KEY,
        region=AZURE_SPEECH_REGION
    )

    speech_config.speech_recognition_language = "ar-EG"

    audio_config = speechsdk.audio.AudioConfig(filename=audio_file)

    recognizer = speechsdk.SpeechRecognizer(
        speech_config=speech_config,
        audio_config=audio_config
    )

    all_text = []
    done = False

    def handle_result(evt):
        if evt.result.text:
            print("📝", evt.result.text)
            all_text.append(evt.result.text)

    def stop(evt):
        nonlocal done
        done = True

    recognizer.recognized.connect(handle_result)
    recognizer.session_stopped.connect(stop)
    recognizer.canceled.connect(stop)

    recognizer.start_continuous_recognition()

    while not done:
        time.sleep(0.5)

    recognizer.stop_continuous_recognition()

    return " ".join(all_text)


## SPLIT TEXT (CHUNKING)

In [7]:
def split_text(text, max_length=4000):
    words = text.split()
    chunks, current_chunk = [], []
    current_length = 0

    for word in words:
        if current_length + len(word) > max_length:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_length = 0

        current_chunk.append(word)
        current_length += len(word) + 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


##  TRANSLATION

In [8]:
def translate_text(text, to_lang="en"):
    url = TRANSLATOR_ENDPOINT + "/translate?api-version=3.0&to=" + to_lang

    headers = {
        'Ocp-Apim-Subscription-Key': TRANSLATOR_KEY,
        'Ocp-Apim-Subscription-Region': TRANSLATOR_REGION,
        'Content-type': 'application/json'
    }

    body = [{'text': text}]

    response = requests.post(url, headers=headers, json=body)
    return response.json()[0]['translations'][0]['text']

def translate_long_text(text, to_lang="en", chunk_size=4000):
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    final_text = ""

    for i, chunk in enumerate(chunks):
        print(f"🌍 Translating part {i+1}/{len(chunks)}...")
        final_text += translate_text(chunk, to_lang) + " "

    return final_text

## SUMMARIZE LONG TEXT

In [9]:
client = TextAnalyticsClient(
    endpoint=AZURE_LANGUAGE_ENDPOINT,
    credential=AzureKeyCredential(AZURE_LANGUAGE_KEY)
)

def summarize_text(text):
    poller = client.begin_extract_summary([text])
    result = poller.result()

    summary = ""
    for doc in result:
        for sentence in doc.sentences:
            summary += sentence.text + " "

    return summary

def summarize_long_text(text):
    chunks = split_text(text)
    final_summary = ""

    for i, chunk in enumerate(chunks):
        print(f"🔹 Summarizing part {i+1}/{len(chunks)}...")
        final_summary += summarize_text(chunk) + "\n"

    return final_summary



## FULL PIPELINE

In [13]:
from google.colab import files

def full_pipeline(video_url):

    if os.path.exists("video.mp4"):
        os.remove("video.mp4")
    if os.path.exists("audio.wav"):
        os.remove("audio.wav")

    print("📥 Downloading video...")
    download_video(video_url)

    print("🎧 Extracting audio...")
    extract_audio("video.mp4", "audio.wav")

    print("🧠 Speech to Arabic text...")
    arabic_text = speech_to_text_long("audio.wav")

    # ✅ حفظ التفريغ
    with open("transcript.txt", "w", encoding="utf-8") as f:
        f.write(arabic_text)

    print("🌍 Translating to English...")
    english_text = translate_long_text(arabic_text, "en")

    print("✍️ Summarizing English text...")
    summary_en = summarize_long_text(english_text)

    print("🔁 Translating summary back to Arabic...")
    summary_ar = translate_long_text(summary_en, "ar")

    # ✅ حفظ الملخص
    with open("summary.txt", "w", encoding="utf-8") as f:
        f.write(summary_ar)

    print("📥 Downloading files...")

    # ✅ تحميل الملفات تلقائي
    files.download("transcript.txt")
    files.download("summary.txt")

    return summary_ar


## vidio link

In [14]:
video_url = "https://youtu.be/MIRUTjjD5lg"

result = full_pipeline(video_url)

print("\n====================")
print("📌 FINAL SUMMARY:")
print("====================\n")
print(result)


📥 Downloading video...
[youtube] Extracting URL: https://youtu.be/MIRUTjjD5lg
[youtube] MIRUTjjD5lg: Downloading webpage


[youtube] MIRUTjjD5lg: Downloading android vr player API JSON
[info] MIRUTjjD5lg: Downloading 1 format(s): 18
[download] Destination: video.mp4
[download] 100% of   39.26MiB in 00:00:01 at 30.21MiB/s  
✅ Video downloaded!
🎧 Extracting audio...
MoviePy - Writing audio in audio.wav


MoviePy - Done.
✅ Audio extracted!
🧠 Speech to Arabic text...
📝 السلام عليكم ورحمه الله وبركاته، بسم الله والحمد لله والصلاه والسلام على رسول الله وبعد ايه الاخبار؟ عاملين ايه يا شباب؟
📝 احنا النهارده في اخر لقاء من عادات العقل بنتكلم عن اخر عادتين عقليتين.
📝 يعني الله يبشرك. خلصنا البرنامج بتاعنا.
📝 ما شاء الله. زي ما جاهزين نبدأوا.
📝 طيب.
📝 النهاردة هنتكلم عن عتيمة عادات العقل. العادة الأولى اسمها التفكير والتواصل بوضوح، والعادة التانية اسمها تحر الدقة. طيب تعالوا نركز كده مع بعض. احنا احنا سبق لينا اتكلمنا عن مجموعة عادات عقلية أخدنا 14 عادة والنهاردة بنكمل آخر عتيمة عادات العقل وهما التفكير والتواصل بوضوح وتحري الدقة.
📝 طيب، تعال نعرف.
📝 يعني ايه التواصل والتفكير بوضوح؟ يعني ايه تحري الدقة؟ باختصار كده يعني لو احنا بنشرحها بالعامية إن أنت لما بتقدر تتكلم أو تفكر بتعرف تفكر بطريقة واضحة للقدامك، يعني نقدر نفهم إنت فكرت إزاي؟ ولما بنسمعك نقدر نفهم أنت عايز إيه بالزبط، إيه اللي أنت محتاجه بالزبط؟
📝 التحدي في الموضوع ده يا شباب.
📝 الكلام نفسه.
📝 آه، أحيانا مبيوصلش لجوانا يعني ممكن مثل 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📌 FINAL SUMMARY:

اليوم، في اللقاء الأخير لعادات العقل، نتحدث عن آخر عادتين من العقل. اليوم سنتحدث عن غموض عادات العقل. قد يعني أنك كما فهمت، أنك ترافقهم لكنك قريب جدا منهم، تبقى كل يوم عائدا إلى منزلهم، وكل يوم يأتون إليك ويتحدثون مع بعضهم البعض، لذا قد يقصدون أنك لا ترافق الناس أكثر من الأذن،  أعني أنك لا تبقى كل يوم برؤية نفس صديقك، والخروج معه كنوع من الخلاص يعني أنك لا تقترب كثيرا، فمن الممكن أن يفهم شخص ما الأمر غير ما فهمته. 
جو دافئ، أعني، ما أريد قوله هو أن التواصل والتحدث مع بعضنا لساعات يحمل أكثر من معنى. لهذا السبب أول مهارة تسمى التفكير والتواصل بوضوح هي أن تسأل الشخص أمامك سؤالا، أليس كذلك؟ عندما تقول إنني شخص جيد، الحمد لله، أنا أول مهارة تسمى القدرة على التفكير والتواصل بوضوح، فهذا يعني أنني أعبر بوضوح عما أريد فعله، حسنا. 
قل كلمة واحدة تعبر عن معنى الدقة. إذا كنت أخا يرتدي بدلة لا أحبها، وأريد أن أخبره برأيي دون أن أؤذيه، يجب أن أقول له: ماذا؟ من المفترض أن أقول له أول شيء أمدحه يعني ماذا أقول له؟ 
أنتم شعب سيء، فلا تخافوا من أن نبيتنا (صلى الله عليه وسلم) ركع عند با